# 19 — Cleanup (GCP)

**AWS-side equivalent:** resource tracker → CDK destroy + manual sweeps (`notebooks/aws/19-cleanup.ipynb`)
**GCP service:** `cleanup_utils.ResourceTracker` walks `config.json['gcp']['tracked_resources']` and calls `gcloud delete` per resource kind.

Pattern: re-read tracker → dry-run → execute → orphan scan → verify.

In [ ]:
# State plumbing — every notebook reads/writes config.json under the 'gcp' namespace.
import json
from pathlib import Path

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
print(f'project={PROJECT_ID} region={REGION}')

# Safety switch — flip to True to actually run the gcloud deletes.
EXECUTE_TEARDOWN = False

## Step 1: Re-read the tracker

`ResourceTracker` reads `config.json` on construction — no surprises across notebook restarts. The list is in append order; teardown walks it in reverse so dependent resources are dropped first (e.g. AlloyDB instances before their cluster — though for AlloyDB, `clusters delete --force` already cascades).

In [ ]:
from cleanup_utils import ResourceTracker, DELETERS

tracker = ResourceTracker()
items = tracker.list()

print(f'Tracked resources: {len(items)}')
by_kind: dict[str, list[dict]] = {}
for r in items:
    by_kind.setdefault(r.get('kind', '?'), []).append(r)
for kind, rows in by_kind.items():
    print(f'\n  {kind} ({len(rows)})')
    for r in rows:
        extra = {k: v for k, v in r.items() if k not in ('kind', 'name')}
        print(f'    - {r.get("name")}  {extra or ""}')

missing_handlers = {k for k in by_kind if k not in DELETERS}
if missing_handlers:
    print(f'\n⚠ no deleter registered for: {sorted(missing_handlers)}')
    print('  These will appear under "skipped" in the teardown summary.')

## Step 2: Dry-run

`teardown_all(dry_run=True)` walks the same list and prints the `gcloud …` command it *would* run, without firing it. Safe to re-run.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
dry = tracker.teardown_all(dry_run=True)
print(f"\nDry-run summary: ok={dry['ok']}  failed={len(dry['failed'])}  skipped={len(dry['skipped'])}")
if dry['skipped']:
    print('\nSkipped (no deleter):')
    for r in dry['skipped']:
        print(f'  - {r}')

## Step 3: Execute teardown (gated)

Flip `EXECUTE_TEARDOWN = True` at the top of the notebook to actually run the `gcloud delete` commands. The teardown is **destructive and idempotent**: it walks the list in reverse insertion order, untracks each resource as it's deleted, and continues past individual failures.

In [ ]:
if EXECUTE_TEARDOWN:
    result = tracker.teardown_all(dry_run=False)
    print(f"\nTeardown summary: ok={result['ok']}  failed={len(result['failed'])}  skipped={len(result['skipped'])}")
    if result['failed']:
        print('\nFailures:')
        for r in result['failed']:
            print(f'  - {r.get("kind")} {r.get("name")}: {r.get("stderr", "")[:200]}')
else:
    print('⏸ EXECUTE_TEARDOWN is False — set it to True at the top of this notebook to actually delete resources.')

## Step 4: Orphan scan

Even with a careful tracker, things slip through (manual `gcloud` runs, half-completed notebooks). After teardown we ask GCP itself what's left with the `orgiq-` prefix in this project + region. Anything that survived is either still tracked (failed delete) or untracked (manual cleanup).

We scan the kinds the tracker manages: AlloyDB clusters, Vertex endpoints, Discovery Engine apps, GCS buckets, reasoning engines (Agent Engine), and Model Armor templates. Each scan is a thin `gcloud … list` filtered by the prefix.

In [ ]:
import subprocess

PREFIX = gcp.get('resource_prefix', 'orgiq')

def _gcloud(args: list[str]) -> tuple[int, str]:
    out = subprocess.run(['gcloud', *args], capture_output=True, text=True)
    return out.returncode, (out.stdout or out.stderr or '').strip()

SCANS = [
    ('alloydb_cluster',
     ['alloydb', 'clusters', 'list', '--region', REGION,
      '--filter', f'name~{PREFIX}', '--format=value(name)']),
    ('vertex_endpoint',
     ['ai', 'endpoints', 'list', '--region', REGION,
      '--filter', f'displayName~{PREFIX}', '--format=value(name)']),
    ('agent_engine',
     ['ai', 'reasoning-engines', 'list', '--region', REGION,
      '--filter', f'displayName~{PREFIX}', '--format=value(name)']),
    ('gcs_bucket',
     ['storage', 'buckets', 'list', '--filter', f'name~{PREFIX}', '--format=value(name)']),
    ('model_armor_template',
     ['model-armor', 'templates', 'list', '--location', REGION,
      '--filter', f'name~{PREFIX}', '--format=value(name)']),
]

remaining_tracked = {(r['kind'], r['name']) for r in tracker.list()}
orphans = []
for kind, args in SCANS:
    rc, out = _gcloud(args)
    if rc != 0:
        print(f'  {kind}: scan failed ({out[:80]}) — skipping')
        continue
    names = [n.split('/')[-1] for n in out.splitlines() if n.strip()]
    if not names:
        print(f'  {kind}: ✓ none remaining')
        continue
    for n in names:
        status = 'tracked (failed delete?)' if (kind, n) in remaining_tracked else 'ORPHAN (not in tracker)'
        print(f'  {kind}: {n} — {status}')
        if (kind, n) not in remaining_tracked:
            orphans.append({'kind': kind, 'name': n})

print(f'\nOrphans not in tracker: {len(orphans)}')

## Step 5: (Optional) Sweep orphans

If the scan turned up resources the tracker doesn't know about, you can re-track them and run teardown again. This is intentionally manual — never auto-delete things the tracker didn't create. Inspect the orphan list, decide what's safe to drop, then re-track + re-tear.

In [ ]:
if orphans:
    print('To re-track and delete an orphan, run:')
    print("  for orph in orphans:")
    print("      tracker.track(orph['kind'], orph['name'], region=REGION)")
    print("  tracker.flush()")
    print("  tracker.teardown_all(dry_run=False)   # set EXECUTE_TEARDOWN=True first")
else:
    print('No orphans — nothing to sweep.')

## Step 6: Final verification

Re-instantiate the tracker (re-reads `config.json`) and confirm it's empty after a successful teardown. The persisted ledger is the source of truth for whether the project is clean.

In [ ]:
final = ResourceTracker().list()
if not final:
    print('✓ tracker is empty — teardown clean')
else:
    print(f'⚠ {len(final)} resources still tracked:')
    for r in final:
        print(f'  - {r}')

## Cost-aware reminders

Resources that bill while idle on GCP (mirror of AWS-side aws/19-cleanup):

| Resource | Idle cost | Mitigation |
|---|---|---|
| **AlloyDB cluster + primary instance** | ~$0.30–0.50/hr depending on tier | `alloydb clusters delete --force` (cascades) |
| **Vertex AI Endpoint** with a deployed model | per-node-hour while up | undeploy model, then `ai endpoints delete` |
| **Agent Engine reasoning engine** | per-invocation only — no idle cost | safe to leave, but delete during teardown |
| **GCS buckets** | per-GB stored | `storage rm --recursive gs://…` |
| **Discovery Engine app + data store** | minor monthly | `alpha discovery-engine apps delete` |
| **Model Armor templates** | none directly; per-request when used | `model-armor templates delete` |

Done. This concludes the GCP notebook track.